In [1]:
!pip install lightgbm scikit-learn

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Running setup.py clean for cuml
Failed to build cuml


  DEPRECATION: Building 'cuml' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'cuml'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> [63 lines of output]
      C:\Users\Shaunak\anaconda3\Lib\site-packages\setuptools\dist.py:765: SetuptoolsDeprecationWarning: License classifiers are deprecated.
      !!
      
              ********************************************************************************
              Please consider removing the following classifiers in favor of a SPDX license expression:
      
              License :: OSI Approved :: A

## MODELS

In [ ]:
models = {
    "lightgbm": {
        "role": "primary_predictor",
        "task": "binary_classification",
        "target": "deteriorates_within_90_days",
        "horizon_days": 90,
        "params": {
            "objective": "binary",
            "n_estimators": 400,
            "learning_rate": 0.05,
            "max_depth": 4,
            "num_leaves": 15,
            "min_child_samples": 50,
            "subsample": 0.8,
            "subsample_freq": 1,
            "colsample_bytree": 0.8,
            "reg_lambda": 1.0,
            "random_state": 42,
            "n_jobs": -1,
            "verbosity": -1,
            "monotone_constraints": {"missed_instalment_intensity": 1},
        },
        "training": {
            "split": "time_based",
            "early_stopping_rounds": 30,
            "imbalance_strategy": "scale_pos_weight = n_negative / n_positive, computed from each training fold",
            "missing_values": "keep_as_null",
            "include_features": [
                "RBDI",
                "CFSI",
                "LPI",
                "SME_index",
                "data_completeness_score",
                "underlying_features",
            ],
        },
        "calibration": {"method": "isotonic", "fit_on": "validation_set"},
        "integration": "Use probability alongside rule indices, exception downgrade, and confidence cap; it does not determine the tier alone.",
    },
    "logistic_regression": {
        "role": "baseline_challenger_scorecard",
        "task": "binary_classification",
        "target": "deteriorates_within_90_days",
        "input_features": "woe_binned",
        "params": {
            "solver": "lbfgs",
            "max_iter": 1000,
            "class_weight": "balanced",
            "random_state": 42,
        },
        "training": {"split": "same_time_based_split_as_lightgbm"},
    },
    "markov_transition": {
        "role": "roll_forward_risk",
        "method": "segment_specific_transition_matrix",
        "states": ["Current", "1-30 DPD", "31-60 DPD", "61-90 DPD", "NPA (90+ DPD)"],
        "segment_by": ["product", "vintage"],
        "prediction_horizons_days": [30, 60],
        "feature_adjustment": "customer_features",
        "output": "next_period_state_probabilities",
    },
    "shap": {
        "role": "lightgbm_explanations_and_reason_codes",
        "model": "lightgbm",
        "output": "per_prediction_feature_contributions",
    },
}

evaluation = {
    "classification_metrics": ["pr_auc", "roc_auc"],
    "calibration_plot": True,
    "lead_time_comparison": "against_dpd_based_detection",
    "false_alarm_comparison": "before_and_after_exception_engine",
}

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression

estimators = {}

for model_name, config in models.items():
    if model_name == "lightgbm":
        estimators[model_name] = LGBMClassifier(**config["params"])
    elif model_name == "logistic_regression":
        estimators[model_name] = LogisticRegression(**config["params"])
    else:
        print(f"{model_name} is a model component, not a scikit-learn estimator.")

print("Initialized estimators:", ", ".join(estimators))